# DriveGuard - Milestone 3 bake-off (Kaggle GPU)

Runs rolling-feature engineering + the model bake-off on Kaggle's GPU/RAM, logs to MLflow,
and saves artifacts to `/kaggle/working` for download back into the repo.

**Before running, in the notebook settings (right panel):**
1. **Add data** -> search your dataset `driveguard-backblaze-interim` -> Add.
2. **Accelerator** -> GPU (T4 x2 or P100).
3. **Internet** -> On.
4. **Add-ons -> Secrets** -> add a secret named `GITHUB_TOKEN` = a GitHub personal access token with `repo` scope (needed to clone the private repo).


In [ ]:
# 1. Clone the private repo using the GITHUB_TOKEN secret
import os, subprocess
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
URL = f'https://{token}@github.com/keerthirevanth/driveguard-predictive-maintenance.git'
REPO = '/kaggle/working/driveguard-predictive-maintenance'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '--depth', '1', URL, REPO], check=True)
print('cloned:', os.path.exists(REPO))

In [ ]:
# 2. Install deps not already on Kaggle
!pip install -q polars pyarrow mlflow optuna pytorch-tabnet 2>/dev/null
import sys
sys.path.insert(0, f'{REPO}/src')
print('python path set')

In [ ]:
# 3. Link the Kaggle dataset into the repo data dirs (robust: recursive + clears stale links)
import os, glob
os.makedirs(f'{REPO}/data/interim', exist_ok=True)
os.makedirs(f'{REPO}/data/processed', exist_ok=True)

quarter_files = glob.glob('/kaggle/input/**/data_*.parquet', recursive=True)
summary_files = glob.glob('/kaggle/input/**/drive_summary.parquet', recursive=True)
assert quarter_files, 'No data_*.parquet under /kaggle/input - is the dataset attached?'
assert summary_files, 'drive_summary.parquet not found under /kaggle/input'

def link(src, dst):
    if os.path.lexists(dst):      # lexists catches broken symlinks too
        os.remove(dst)
    os.symlink(src, dst)

for f in quarter_files:
    link(f, f"{REPO}/data/interim/{os.path.basename(f)}")
link(summary_files[0], f'{REPO}/data/processed/drive_summary.parquet')

print('interim:', sorted(os.listdir(f'{REPO}/data/interim')))
print('processed:', os.listdir(f'{REPO}/data/processed'))

In [ ]:
# 4. Build feature sets for N=30: point-in-time (big5) and rolling/temporal
from pathlib import Path
from driveguard.config import load_config
from driveguard.features.build_features import make_dataset
from driveguard.features.rolling import make_rolling_dataset
ROOT = Path(REPO)
cfg = load_config(f'{REPO}/config/config.yaml')
N = 30
print('building big5 (point-in-time)...'); make_dataset(cfg, ROOT, N, 'big5')
print('building rolling (temporal)...'); make_rolling_dataset(cfg, ROOT, N)
print('done')

In [ ]:
# 5. Bake-off: same lineup on both feature sets, logged to MLflow
import json
from driveguard.models.train import run_bakeoff, FACTORY
MLRUNS = '/kaggle/working/mlruns'
MODELS = ['logreg', 'random_forest', 'lightgbm', 'xgboost', 'catboost', 'tabnet']
all_results = []
for fset in ['big5', 'rolling']:
    fdir = f'{REPO}/data/processed/features_{fset}_N{N}'
    board = run_bakeoff(fdir, fset, N, MODELS, cfg, MLRUNS)
    all_results += board
    Path(f'{REPO}/reports').mkdir(exist_ok=True)
    json.dump(board, open(f'/kaggle/working/bakeoff_{fset}_N{N}.json', 'w'), indent=2)
print('bake-off complete')

In [ ]:
# 6. Leaderboard (test PR-AUC vs the M2 baseline of 0.096)
import pandas as pd
rows = []
for r in all_results:
    if r.get('status') == 'ok':
        rows.append({'model': r['model'], 'feature_set': r['feature_set'],
                     'test_pr_auc': round(r['test']['pr_auc'], 4),
                     'test_roc_auc': round(r['test']['roc_auc'], 3),
                     'test_recall@1%fpr': round((r['test']['recall_at_fpr_1pct']['recall'] or 0), 3),
                     'fit_sec': r.get('fit_sec')})
lb = pd.DataFrame(rows).sort_values('test_pr_auc', ascending=False)
print('M2 baseline test PR-AUC = 0.096')
lb

In [ ]:
# 7. Package artifacts for download (Save Version -> Output persists these)
import shutil
shutil.make_archive('/kaggle/working/mlruns', 'zip', '/kaggle/working/mlruns')
print('artifacts in /kaggle/working:')
!ls -lh /kaggle/working/*.json /kaggle/working/mlruns.zip